<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/notebooks/stage_05_time_aware_data_splitting/stage_05_time_aware_data_splitting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **stage_05_time_aware_data_splitting**

## Introducción y Resumen

Esta notebook tiene como objetivo dividir el dataset MNQ en conjuntos de entrenamiento, validación y prueba, asegurando una partición aleatoria, reproducible y estructuralmente consistente. Con ello se dejan listos los datos para el entrenamiento y evaluación de los modelos predictivos.

0. Configuración del Entorno

    Se conecta Google Drive y se clona el repositorio de trabajo. Se instalan e importan librerías necesarias como pandas, numpy, matplotlib y seaborn. Se cargan los datasets procesados previamente (mnq_technical_indicators y mnq_alpha_factors) y se muestra un resumen de la información del dataset MNQ.

1. Carga de datos

    Se importa el dataset procesado con features técnicos y alpha factors. Se revisa su estructura (filas, columnas, tipos de datos) y también de importa el listado de features seleccionados para cada ventana de tiempo.

2. Análisis del dataset `mnq_model`

    Se revisa la estructura del dataset (filas, columnas, tipos de datos), se busca los valores NaNs y se verifica la distribución temporal de los registros.

3. Definición de parámetros de división

    En este punto se define la estrategia de partición del dataset: se toma un 70% de los días para entrenamiento, y el 30% restante se divide en partes iguales para validación y prueba. De esta manera, el modelo cuenta con suficientes datos para aprender, mientras que se reservan bloques temporales separados para ajustar parámetros y evaluar el rendimiento final sin fugas de información.

4. Selección aleatoria de días.

    Este punto busca garantizar que la partición de los datos sea representativa y no esté sesgada por la secuencia temporal. Al asignar los días de forma aleatoria —aunque de manera reproducible— se evita que los conjuntos queden condicionados por períodos específicos del mercado (por ejemplo, tendencias prolongadas o alta volatilidad en ciertos meses). Así, cada subconjunto refleja mejor la diversidad del dataset y se obtiene una evaluación más robusta del modelo.

5. Generación de datasets `mnq_train`, `mnq_test` y `mnq_valid`

    En este punto se crean los datasets mnq_train, mnq_valid y mnq_test, manteniendo homogeneidad en estructura (301 registros por día, de 09:30 a 14:30) y sin solapamiento entre conjuntos. Esto asegura consistencia en el entrenamiento, validación y prueba del modelo.

## 0. Configuración del Entorno


### 0.1. Clonado de repositorio / Acceso a Drive

In [25]:
#Clonamos el repo
#LINK DE REPOSITORIO: https://github.com/GUNAPILLCO/neural_profit
#!git clone https://github.com/GUNAPILLCO/neural_profit.git

In [2]:
from google.colab import drive
drive.mount('/content/drive')
drive_path = "/content/drive/MyDrive/neural_profit"

Mounted at /content/drive


### 0.2. Instalación de librerías


In [3]:
#!{sys.executable} -m pip install -q ta
#print("Librería instalada: technical-analysis")

### 0.3. Importación de librerías


In [4]:
import sys
import re
#Instalación de librería pandas_market_calendars
#!{sys.executable} -m pip install -q pandas_market_calendars
#print("Librería instalada: pandas_market_calendars")


from functools import reduce
# Utilidades generales
from datetime import datetime, timedelta
import os
import glob
import requests
import warnings
warnings.filterwarnings('ignore')

# Manejo y procesamiento de datos
#import ta
import pandas as pd
import numpy as np
from tabulate import tabulate
import matplotlib.pyplot as plt
# Calendario de mercados
#import pandas_market_calendars as mcal

#from ta.momentum import StochasticOscillator, ROCIndicator
#from ta.volatility import BollingerBands, AverageTrueRange

from scipy.stats import spearmanr

import os
import json
import logging
from pathlib import Path
from typing import Dict, Any, List, Tuple

import numpy as np
import pandas as pd

#from ta.momentum import ROCIndicator

# ----------------------------
# Logging
# ----------------------------
logging.basicConfig(
    level=os.environ.get("LOG_LEVEL", "INFO"),
    format="%(asctime)s | %(levelname)s | %(message)s",
)
log = logging.getLogger("stage_05_time_aware_data_splitting")

In [5]:
# ============================================================
# Paths / IO (via env o defaults)
# ============================================================

DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))


IN_PARQUET = Path(os.environ.get("IN_PARQUET", "data/features/mnq_features_target.parquet"))

OUT_SPLITS = Path(os.environ.get("OUT_PARQUET", "data/splits/splits.json"))
OUT_PARQUET_TRAIN = Path(os.environ.get("OUT_PARQUET", "data/splits/mnq_train.parquet"))
OUT_PARQUET_VALID = Path(os.environ.get("OUT_PARQUET", "data/splits/mnq_valid.parquet"))
OUT_PARQUET_TEST = Path(os.environ.get("OUT_PARQUET", "data/splits/mnq_test.parquet"))

OUT_SUMMARY = Path(os.environ.get("OUT_SUMMARY", "reports/splits_summary.json"))


# Si en su pipeline hay un artifact de targets, puede quedar declarado,
# pero NO es estrictamente necesario en stage_04.
IN_ARTIFACT = Path(os.environ.get("IN_ARTIFACT", "reports/features_target_summary.json"))

# PARA EL NOTEBOOK:
IN_PARQUET = DRIVE_DIR / IN_PARQUET
IN_ARTIFACT = DRIVE_DIR / IN_ARTIFACT

OUT_SPLITS = DRIVE_DIR / OUT_SPLITS
OUT_PARQUET_TRAIN = DRIVE_DIR / OUT_PARQUET_TRAIN
OUT_PARQUET_VALID = DRIVE_DIR / OUT_PARQUET_VALID
OUT_PARQUET_TEST = DRIVE_DIR / OUT_PARQUET_TEST
OUT_SUMMARY = DRIVE_DIR / OUT_SUMMARY


## **1. Carga de datos**

### 1.1. Carga de dataset `mnq_features_target.parquet`




In [6]:
def _ensure_parent_dir(path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)


# ============================================================
# 1) Carga y preprocesamiento base
# ============================================================
def load_mnq_parquet(path: Path = IN_PARQUET) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el parquet de entrada: {path}")
    log.info(f"[OK] Cargando parquet: {path}")
    return pd.read_parquet(path)

def add_column_date(df: pd.DataFrame, date_col: str = "date") -> pd.DataFrame:
    """
    Asegura DatetimeIndex y agrega columna 'date' (YYYY-MM-DD) para agrupar por jornada.
    """
    out = df.copy()
    out.index = pd.to_datetime(out.index)
    out[date_col] = out.index.date
    # Reordenar (date primero)
    cols = [date_col] + [c for c in out.columns if c != date_col]
    return out[cols]


In [7]:
mnq_features_target = load_mnq_parquet(IN_PARQUET)

### 1.2. Información de dataset MNQ_to_model


In [8]:
def info_dataset(df, name: str):
  print(f"Información del dataset {name}:\n")

  # Contar valores únicos en la columna 'date'
  num_dias = df['date'].nunique()
  print(f"\tCantidad de días: {num_dias}")

  # Filtrar valores válidos
  validos_por_dia = df.dropna(subset=['close']).groupby('date').size()

  # Calcular el promedio
  promedio_por_fecha = validos_por_dia.mean()
  print(f"\tRegistros por día: {int(promedio_por_fecha)}")

  primer_hora = df.index[0].strftime('%H:%M')
  ultima_hora = df.index[-1].strftime('%H:%M')
  zona_horaria = df.index[0].tzinfo


  print(f"\tHora diaria de inicio {primer_hora}")
  print(f"\tHora diaria de final {ultima_hora}")
  print(f"\tZona horaria: {zona_horaria}\n")

  return num_dias, promedio_por_fecha

In [9]:
num_dias, promedio_por_fecha = info_dataset(mnq_features_target, 'mnq_features_target')

Información del dataset mnq_features_target:

	Cantidad de días: 1303
	Registros por día: 421
	Hora diaria de inicio 07:30
	Hora diaria de final 14:30
	Zona horaria: America/New_York



### 1.3. Carga de listado de features y targets

In [10]:
import json
#/content/drive/MyDrive/neural_profit/reports/features_target_summary.json
with open(IN_ARTIFACT, "r", encoding="utf-8") as f:
    features_target_summary = json.load(f)

features = features_target_summary["schema"]["features"]
targets = features_target_summary["schema"]["targets"]

print("Features:", features)
print("Targets:", targets)

Features: ['open', 'high', 'low', 'close', 'price_ema60', 'momentum_10', 'roc_30', 'roc_60']
Targets: ['delta_pts_60', 'delta_pts_90']


## **2. Análisis del dataset `mnq_features_target`**

### 2.0. Funciones

#### Función para contar NaN en dataset

In [29]:
import pandas as pd

def nan_count(df: pd.DataFrame) -> pd.DataFrame:
    """
    Verifica la existencia de NaN por día y por columna.
    Si detecta algún NaN, loguea el detalle y corta la ejecución del script.
    """

    log.info("[2.1] Iniciando validación de NaN por día y por columna")

    # Conteo de NaN por día y columna
    daily_nan_counts = (
        df.groupby("date")
        .apply(lambda x: x.isna().sum())
    )

    # Máscara de presencia de NaN
    nan_mask = daily_nan_counts > 0

    if nan_mask.any().any():
        log.info("[ERROR] Se detectaron valores NaN en el dataset")

        # Detalle de NaN encontrados
        nan_details = (
            daily_nan_counts[nan_mask]
            .stack()
            .reset_index()
            .rename(columns={
                "level_1": "column",
                0: "nan_count"
            })
        )

        log.info("[RROR] Detalle de NaN encontrados (fecha, columna, cantidad):")
        log.info("\n%s", nan_details.to_string(index=False))

        # Corte inmediato del script
        raise RuntimeError("[ERROR] Ejecución detenida por presencia de NaN en el dataset")

    log.info("[OK] Validación NaN OK: no se encontraron valores faltantes")

    # Resumen (control)
    daily_unique_nans = pd.DataFrame({
        "feature": daily_nan_counts.columns,
        "daily_nan_counts": [
            sorted(daily_nan_counts[col].unique())
            for col in daily_nan_counts.columns
        ]
    })

    return daily_unique_nans


#### Función para detectar saltos temporales (gaps)

In [12]:
def detectar_gaps(df: pd.DataFrame, gap_minutes: int = 1):
    """
    Verifica si existen saltos mayores al intervalo esperado (por defecto 1 minuto)
    entre registros consecutivos dentro de cada día, en un DataFrame con índice tipo DatetimeIndex.

    Omite el primer registro de cada día.

    Parámetros:
    - df: DataFrame con índice datetime.
    - gap_minutes: tamaño esperado del intervalo en minutos (por defecto 1).

    Retorna:
    - Lista de índices donde se detectaron diferencias mayores al intervalo esperado.
    """
    df = df.copy()
    df['time_diff'] = df.index.to_series().diff()

    base_time_diff = pd.Timedelta(minutes=gap_minutes)
    problem_indices = []

    for date, group in df.groupby(df.index.date):
        time_diff = group['time_diff'].iloc[1:]
        incorrect_indices = time_diff[time_diff != base_time_diff].index
        if len(incorrect_indices) > 0:
            problem_indices.append(incorrect_indices)

    if problem_indices:
        print(f"Se encontraron problemas en {len(problem_indices)} registros con diferencias irregulares.\n")

        # Conteo por fecha
        conteos = df.groupby(df.index.date).size()

        for i in range(len(problem_indices)):
            idx = problem_indices[i][0]
            diff = df.loc[idx, 'time_diff']
            date = idx.date()
            count = conteos[date]
            print(f'\t{idx} -> Diferencia: {diff} | # Registros: {count}')
    else:
        print("No se encontraron problemas, todas las muestras son consecutivas minuto a minuto.")

    return problem_indices

### 2.1. Búsqueda de NaN en `mnq_model`:

Buscamos los valores NaN en todas la columnas del dataset:

In [28]:
mnq_nans = nan_count(mnq_features_target)


### 2.2. Búsqueda de gaps en `mnq_model_clean`

In [14]:
detectar_gaps(mnq_features_target)

No se encontraron problemas, todas las muestras son consecutivas minuto a minuto.


[]

Contamos con el dataset limpio de NaNs y saltos temporales.

## **3. Definición de parámetros de división**

Para dividir el dataset en subconjuntos, se utiliza la siguiente estrategia:

- 70% de los días se asignan al conjunto de entrenamiento (train).

- El 30% restante se reparte de manera equitativa entre los conjuntos de validación (valid) y prueba (test).

Esto garantiza que el modelo disponga de la mayor parte de los datos para aprender patrones, mientras que las particiones de validación y prueba permiten ajustar hiperparámetros y evaluar el rendimiento fuera de muestra.

De esta forma, se asegura un esquema de división temporalmente consistente, sin solapamiento entre conjuntos.

In [15]:
n_train = int(num_dias * 0.7)
n_valid = int(num_dias * 0.15)
n_test  = num_dias - n_train - n_valid

In [16]:
print(f'n_train: {n_train}')
print(f'n_valid: {n_valid}')
print(f'n_test: {n_test}')

n_train: 912
n_valid: 195
n_test: 196


##**4. Partición del dataset respetando causalidad temporal**

En este punto, la división del dataset se realiza respetando el orden cronológico de los días, con el objetivo de preservar la causalidad temporal y evitar cualquier forma de data leakage en la evaluación del modelo.

Para ello, se extraen los días únicos presentes en el dataset y se ordenan cronológicamente. A continuación, se asignan los primeros días al conjunto de entrenamiento, los días intermedios al conjunto de validación y los días más recientes al conjunto de prueba, de acuerdo con las proporciones definidas (70 % / 15 % / 15 %).

Este procedimiento garantiza que el modelo sea entrenado exclusivamente con información pasada y evaluado sobre datos futuros, manteniendo la integridad intradía de cada jornada y proporcionando una estimación realista de su capacidad de generalización.

In [17]:
# Obtener días únicos y ordenarlos cronológicamente
unique_days = pd.Index(sorted(mnq_features_target["date"].unique()))
num_dias = len(unique_days)

# Dividir días en orden temporal
train_days = unique_days[:n_train]
val_days   = unique_days[n_train:n_train + n_valid]
test_days  = unique_days[n_train + n_valid:]

In [18]:
print("Train days:")
print(f"  Desde: {train_days[0]}")
print(f"  Hasta: {train_days[-1]}")

print("\nValidation days:")
print(f"  Desde: {val_days[0]}")
print(f"  Hasta: {val_days[-1]}")

print("\nTest days:")
print(f"  Desde: {test_days[0]}")
print(f"  Hasta: {test_days[-1]}")

Train days:
  Desde: 2019-12-23
  Hasta: 2023-10-26

Validation days:
  Desde: 2023-10-27
  Hasta: 2024-08-20

Test days:
  Desde: 2024-08-21
  Hasta: 2025-06-13


## **5. Generación de datasets `mnq_train`, `mnq_test` y `mnq_valid`**

En este paso generamos los datasets finales para cada subconjunto: `mnq_train`, `mnq_valid` y `mnq_test`. La asignación se realiza filtrando los días correspondientes a cada conjunto, lo que asegura que no exista solapamiento entre ellos.

In [19]:
mnq_features_target

,date,open,high,low,close,price_ema60,momentum_10,roc_30,roc_60,delta_pts_60,delta_pts_90
datetime,,,,,,,,,,,
2019-12-23 07:30:00-05:00,2019-12-23,8736.00,8736.75,8736.00,8736.75,0.000510,0.000143,0.100252,0.103119,0.25,0.50
2019-12-23 07:31:00-05:00,2019-12-23,8736.00,8736.00,8735.75,8735.75,0.000381,0.000143,0.100264,0.105999,0.75,2.25
2019-12-23 07:32:00-05:00,2019-12-23,8735.50,8735.50,8734.50,8735.00,0.000284,0.000086,0.080202,0.111745,1.00,3.00
2019-12-23 07:33:00-05:00,2019-12-23,8735.00,8735.75,8734.25,8734.50,0.000218,0.000086,0.071607,0.097410,2.25,3.25
2019-12-23 07:34:00-05:00,2019-12-23,8734.50,8734.50,8734.00,8734.00,0.000155,0.000029,0.060146,0.091680,3.75,3.50
...,...,...,...,...,...,...,...,...,...,...,...
2025-06-13 14:26:00-04:00,2025-06-13,21716.50,21722.75,21712.00,21719.50,-0.001801,-0.000207,-0.309818,-0.357839,-90.00,-102.00
2025-06-13 14:27:00-04:00,2025-06-13,21719.00,21719.75,21695.75,21698.50,-0.002675,-0.000714,-0.406206,-0.419917,-69.25,-74.75
2025-06-13 14:28:00-04:00,2025-06-13,21698.25,21700.25,21670.50,21679.25,-0.003444,-0.001577,-0.488852,-0.493419,-52.25,-57.50


In [20]:
# Crear datasets por días completos
mnq_train = mnq_features_target[mnq_features_target["date"].isin(train_days)].copy()
mnq_valid = mnq_features_target[mnq_features_target["date"].isin(val_days)].copy()
mnq_test  = mnq_features_target[mnq_features_target["date"].isin(test_days)].copy()

# (Opcional) Ordenar por tiempo dentro de cada dataset
mnq_train = mnq_train.sort_values(["date"])
mnq_valid = mnq_valid.sort_values(["date"])
mnq_test  = mnq_test.sort_values(["date"])

In [21]:
print("Rows:")
print("Train:", len(mnq_train))
print("Valid:", len(mnq_valid))
print("Test :", len(mnq_test))

print("\nDays:")
print("Train:", mnq_train["date"].nunique())
print("Valid:", mnq_valid["date"].nunique())
print("Test :", mnq_test["date"].nunique())

Rows:
Train: 383952
Valid: 82095
Test : 82516

Days:
Train: 912
Valid: 195
Test : 196


Para un análisis posterior usamos la función antes definida `info_dataset` para confirmar que todos los subconjuntos comparten las mismas características estructurales:

In [22]:
print("Train index:")
print("  Desde:", mnq_train.index.min())
print("  Hasta:", mnq_train.index.max())

print("\nValidation index:")
print("  Desde:", mnq_valid.index.min())
print("  Hasta:", mnq_valid.index.max())

print("\nTest index:")
print("  Desde:", mnq_test.index.min())
print("  Hasta:", mnq_test.index.max())

Train index:
  Desde: 2019-12-23 07:30:00-05:00
  Hasta: 2023-10-26 14:30:00-04:00

Validation index:
  Desde: 2023-10-27 07:30:00-04:00
  Hasta: 2024-08-20 14:30:00-04:00

Test index:
  Desde: 2024-08-21 07:30:00-04:00
  Hasta: 2025-06-13 14:30:00-04:00


Se verifica que todos los datasets poseen una base homogénea, lo que nos va a facilitar la comparación de resultados entre etapas de entrenamiento, ajuste y evaluación. Además, la consistencia en el número de registros por día nos garantiza que los modelos reciban siempre ventanas de información con la misma extensión temporal.

## **6. Guardamos los datasets generados**


In [23]:
#/content/drive/MyDrive/neural_profit/data/splits
OUT_PARQUET_TRAIN

PosixPath('/content/drive/MyDrive/neural_profit/data/splits/mnq_train.parquet')

In [24]:
#Guardamos el dataset
mnq_train.to_parquet(OUT_PARQUET_TRAIN, index=True)
mnq_valid.to_parquet(OUT_PARQUET_VALID, index=True)
mnq_test.to_parquet(OUT_PARQUET_TEST, index=True)

## **7. Para notebook**
